# 10 · Redondeo duro de $W^*$ con restricciones por bloque

El modelo ajustado $W^*$ (perfiles × manzanas, notebook 05) es fraccional. Aquí se busca una matriz entera $Y$ que cumpla **exactamente** todas las restricciones del ajuste: tamaños de perfil $\mu_p$, totales por manzana $\nu_m$ y los marginales publicados de cada bloque por manzana $\nu_{kcm}$, moviéndose a lo más una unidad respecto de $W^*$ en cada celda.

En el notebook 09 se vio que el redondeo por flujo (solo $\mu_p$, $\nu_m$ y cotas) cumple filas y columnas pero rompe los bloques (errores de hasta 2.338 personas por manzana). Esta es la versión dura del problema; la blanda (minimizar la desviación de los bloques) está en `manzanas/redondeo.py` con `duro=False`.

Módulo: `manzanas/redondeo.py`. Corrida larga: `scripts/correr_redondeo.py --duro`.

In [1]:
import os, json, numpy as np
import scipy.sparse as sp
from manzanas.unidades import cargar_unidad
from manzanas.pais import rutas_unidad
from manzanas.perfiles import cargar_modelo
from manzanas import redondeo as rd

cut, unidad = 13108, "personas"
datos, inv = cargar_unidad(cut, unidad)
modelo = cargar_modelo(os.path.join(rutas_unidad(unidad)["modelos"], f"modelo_{cut}.npz"))

W = modelo["W"]                # (perfiles, manzanas)
mu = modelo["mu_p"]            # tamaño de cada perfil
nu = modelo["nu_m"] if modelo["nu_m"] is not None else datos.nu   # total por manzana
ids = datos.manzanas_ids
assert W.shape == (len(mu), len(nu)) and mu.sum() == nu.sum()

F, frac, r, c = rd.partes(W, mu, nu)
print(f"comuna {cut}, {unidad}: W {W.shape}, {len(modelo['bloques'])} bloques")
print(f"sum mu = sum nu = {mu.sum():,} | perfiles con mu_p = 1: {(mu == 1).sum():,}")
print(f"celdas fraccionales: {(frac > rd.EPS).sum():,} | unidades a repartir con z: {r.sum():,}")

comuna 13108, personas: W (40946, 469), 27 bloques
sum mu = sum nu = 116,943 | perfiles con mu_p = 1: 30,742
celdas fraccionales: 13,213,648 | unidades a repartir con z: 90,145


## Planteamiento

Notación:
- $p = 1..P$ perfiles, $m = 1..M$ manzanas, $k = 1..K$ bloques.
- $x_{pk}$: categoría del perfil $p$ en el bloque $k$ (`perfiles[:, k]`).
- $\nu_{kcm}$: marginal del censo, unidades de la categoría $c$ del bloque $k$ en la manzana $m$ (incluye la fila "resto" = total − publicadas).
- $\mathcal{O}_k$: pares $(c, m)$ observados (se excluyen los enmascarados, NaN).

Problema entero:
$$
\begin{aligned}
\text{buscar } & y \in \mathbb{Z}^{P\times M} \quad (\text{con } \min\ f(y),\ f = 0 \text{ o cercanía a } W^*) \\
\text{s.a. } & \textstyle\sum_m y_{pm} = \mu_p && \forall p \\
& \textstyle\sum_p y_{pm} = \nu_m && \forall m \\
& \textstyle\sum_{p:\,x_{pk}=c} y_{pm} = \nu_{kcm} && \forall k,\ (c,m)\in\mathcal{O}_k \\
& \lfloor W^*_{pm}\rfloor \le y_{pm} \le \lceil W^*_{pm}\rceil && \forall p, m
\end{aligned}
$$

Con $y_{pm} = \lfloor W^*_{pm}\rfloor + z_{pm}$ y $z_{pm} \in \{0,1\}$ en las celdas fraccionales, las restricciones quedan en $z$ con lados derechos
$r_p = \mu_p - \sum_m \lfloor W^*_{pm}\rfloor$, $c_m = \nu_m - \sum_p \lfloor W^*_{pm}\rfloor$ y $\nu_{kcm} - \sum_{p:\,x_{pk}=c}\lfloor W^*_{pm}\rfloor$.

Objetivo opcional de cercanía: $\sum |y_{pm} - W^*_{pm}| = \text{cte} + \sum (1 - 2\,\text{frac}_{pm})\, z_{pm}$, lineal en $z$.

**Observaciones:**
1. Sin las restricciones de bloque la matriz es de transporte (totalmente unimodular) y el PL da enteros (notebook 09). Con los bloques ya no: es un problema entero de verdad, y **puede no tener solución** (redondeo controlado multidimensional).
2. Los microdatos reales cumplen las tres igualdades, pero no necesariamente las cotas $\lfloor W^*\rfloor, \lceil W^*\rceil$. Si el problema tiene solución y además hay muchos perfiles únicos ($\mu_p = 1$), una solución puede acercarse mucho a la asignación real: es un ataque de reconstrucción (ver proyecto `census-privacy`).

## Condición necesaria por restricción

Cada restricción de bloque tiene que poder alcanzarse con las celdas libres:
$$\textstyle\sum_{p:\,x_{pk}=c} \lfloor W^*_{pm}\rfloor \;\le\; \nu_{kcm} \;\le\; \sum_{p:\,x_{pk}=c} \lfloor W^*_{pm}\rfloor + \#\{p : x_{pk}=c,\ (p,m) \text{ libre}\}$$

Es necesaria, no suficiente: las restricciones comparten variables.

In [2]:
libres_todas = frac > rd.EPS
print(f"{'celdas libres':28s} {'n':>11s} {'violan':>7s} {'peor':>5s}")
for nombre, libre in [("todas las fraccionales", libres_todas),
                      ("tau=0.001, L=3", rd.celdas_libres(frac, 0.001, 3)),
                      ("tau=0.01, L=3", rd.celdas_libres(frac, 0.01, 3)),
                      ("tau=0.05, L=3", rd.celdas_libres(frac, 0.05, 3))]:
    cn = rd.condicion_necesaria(F, libre, inv, modelo)
    violan = sum(int(((f > 0) | (s > 0)).sum()) for f, s in cn.values())
    peor = max(max(f.max(), s.max()) for f, s in cn.values())
    print(f"{nombre:28s} {libre.sum():11,} {violan:7d} {peor:5.0f}")

celdas libres                          n  violan  peor


todas las fraccionales        13,213,648       0     0


tau=0.001, L=3                 7,047,944       0     0


tau=0.01, L=3                  1,539,453     242     3


tau=0.05, L=3                    336,395    9148    50


## Celdas libres

Con todas las celdas fraccionales son ~13 millones de binarias y ~380 millones de no ceros: demasiado. Se dejan libres las celdas con parte fraccional $\ge \tau$ más las $L$ mayores de cada fila, y se **reparan** las restricciones de bloque que no alcanzan: en cada una se liberan las celdas de mayor fracción del grupo (perfiles con esa categoría en esa manzana) hasta que la condición necesaria se cumpla (`celdas_libres_duras`).

Ojo: aunque pase la condición necesaria, la relajación lineal puede ser infactible. En el notebook 09, la relajación de la versión blanda con $\tau = 0{,}05$ dio una desviación mínima > 100.000 y con $\tau = 0{,}01$ > 260 (antes de reparar): con esas celdas la versión dura no tiene solución ni siquiera fraccional. Mientras más chico $\tau$, más posibilidades de factibilidad, pero más grande el problema.

In [3]:
TAU, L, OBJETIVO = 0.01, 3, "cero"     # OBJETIVO: "cero" (factibilidad) o "cercania"

libre = rd.celdas_libres_duras(F, frac, inv, modelo, TAU, L)
extra = libre.sum() - rd.celdas_libres(frac, TAU, L).sum()
cn = rd.condicion_necesaria(F, libre, inv, modelo)
pi, mi = np.nonzero(libre)
print(f"tau = {TAU}, L = {L}: {libre.sum():,} celdas libres ({extra:,} agregadas por reparación)")
print("condición necesaria tras reparar, restricciones que violan:", sum(int(((f > 0) | (s > 0)).sum()) for f, s in cn.values()))
print("mu/nu factibles con estas celdas (flujo máximo):", rd.flujo(r, c, pi, mi)[1])

mip = rd.construir(W, mu, nu, inv, modelo, libre=libre, duro=True, objetivo=OBJETIVO)
print(f"MIP duro: {mip['A'].shape[0]:,} restricciones ({mip['P']:,} filas + {mip['M']} columnas + {mip['B']:,} de bloque)")
print(f"          {mip['n_z']:,} binarias, {mip['A'].nnz:,} no ceros, objetivo = {OBJETIVO}")

tau = 0.01, L = 3: 1,539,712 celdas libres (259 agregadas por reparación)
condición necesaria tras reparar, restricciones que violan: 0
mu/nu factibles con estas celdas (flujo máximo): True


MIP duro: 91,424 restricciones (40,946 filas + 469 columnas + 50,009 de bloque)
          1,539,712 binarias, 43,868,387 no ceros, objetivo = cero


## Resolver

El MIP es grande; se resuelve fuera del notebook. Dos caminos (mismos $\tau$, $L$, objetivo, para que las celdas libres coincidan):

**A. HiGHS en este servidor** (en segundo plano, sigue aunque se cierre la terminal):
```
nohup env_censo/bin/python scripts/correr_redondeo.py 13108 --duro --tau 0.01 --L 3 --horas 24 > resultados/pais/redondeo/corrida_duro_13108.log 2>&1 &
```
Escribe `resultados/pais/redondeo/Y_personas_13108_duro_t0.01_L3.npz` si encuentra solución, o termina con estado `Infeasible`.

**B. Gurobi en otro computador**: escribir el `.mps` (celda siguiente), copiarlo, resolver y traer el `.sol`:
```
gurobi_cl ResultFile=redondeo_personas_13108_duro_t0.01_L3.sol redondeo_personas_13108_duro_t0.01_L3.mps
```
Si Gurobi declara infactible, `gurobi_cl ResultFile=iis.ilp ...` entrega un subconjunto irreducible de restricciones en conflicto.

In [4]:
ESCRIBIR_MPS = False   # el archivo pesa del orden de 1-2 GB con tau = 0.01

ruta_red = os.path.join("resultados", "pais", "redondeo")
ruta_mps = os.path.join(ruta_red, rd.nombre("redondeo", unidad, cut, True, TAU, L) + ".mps")
if ESCRIBIR_MPS:
    os.makedirs(ruta_red, exist_ok=True)
    print(rd.escribir_mps(mip, ruta_mps), f"{ruta_mps}: {os.path.getsize(ruta_mps) / 1e6:,.0f} MB")
else:
    print("no se escribe el .mps (ESCRIBIR_MPS = False);", "ya existe" if os.path.exists(ruta_mps) else "no existe", ruta_mps)

no se escribe el .mps (ESCRIBIR_MPS = False); no existe resultados/pais/redondeo/redondeo_personas_13108_duro.mps


## Cargar y verificar la solución

Se acepta una solución de HiGHS (`.npz`) o de Gurobi (`.sol`) y se verifica exactamente: filas, columnas, cotas y cada restricción de bloque observada.

In [5]:
ruta_npz = os.path.join(ruta_red, rd.nombre("Y", unidad, cut, True, TAU, L) + ".npz")
ruta_sol = ruta_mps.replace(".mps", ".sol")
if os.path.exists(ruta_sol):
    Y = rd.matriz_Y(mip, rd.leer_sol(mip, ruta_sol)); origen = ruta_sol
elif os.path.exists(ruta_npz):
    Y = sp.load_npz(ruta_npz).toarray(); origen = ruta_npz
else:
    Y, origen = None, None
    resumen = ruta_npz.replace(".npz", ".json")
    print("todavía no hay solución.", "Revisar el log de la corrida." if not os.path.exists(resumen) else open(resumen).read())

if Y is not None:
    print("solución desde", origen)
    for k, v in rd.verificar(mip, Y, mu, nu, inv, modelo).items():
        print(f"  {k}: {v}")
    print(f"max |Y - W*| = {np.abs(Y - W).max():.4f} | sum |Y - W*| = {np.abs(Y - W).sum():,.1f}")

todavía no hay solución. Revisar el log de la corrida.
